<a href="https://colab.research.google.com/github/lubbad-aljazeera/paid-activity-data-pipeline/blob/main/TikTok_Data_Pipeline_Extended.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from google.cloud import storage, bigquery
from google.colab import auth
from datetime import datetime
import re
import os

# 1. AUTHENTICATION & CONFIGURATION
auth.authenticate_user()
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_id = "aj360"
table_id = "tiktok_ads"
bucket_name = "aj360_data"
# Ensure the prefix targets the specific file path in GCS
prefix = "TIKTOK_ADS/tiktok_ads_country_5-7Mar2026.csv"

bq_client = bigquery.Client(project=project_id)
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# ---------------------------------------------------------
# STEP 1: MONTHLY BACKUP MECHANISM
# ---------------------------------------------------------
# Creates a snapshot of the table before any new data is appended
backup_suffix = datetime.now().strftime('%Y_%m_%d').lower()
backup_table_id = f"{table_id}_backup_{backup_suffix}"
backup_ref = f"{project_id}.{dataset_id}.{backup_table_id}"

try:
    bq_client.get_table(table_ref)
    copy_job = bq_client.copy_table(
        table_ref,
        backup_ref,
        job_config=bigquery.CopyJobConfig(write_disposition="WRITE_TRUNCATE")
    )
    copy_job.result()
    print(f"✅ Monthly backup created/updated: {backup_table_id}")
except Exception as e:
    print(f"⚠️ Backup skipped (Table may not exist yet): {e}")

# ---------------------------------------------------------
# STEP 2: LOAD LOOKUP TABLE
# ---------------------------------------------------------
try:
    lookup_query = f"SELECT term, show_title_en FROM `{project_id}.{dataset_id}.paid_ads_shows_lookup`"
    lookup_df = bq_client.query(lookup_query).to_dataframe()
    # Normalize lookup terms to lowercase for case-insensitive matching
    term_lookup = {str(k).lower(): v for k, v in zip(lookup_df['term'], lookup_df['show_title_en'])}
    print(f"✅ Lookup table loaded: {len(term_lookup)} terms.")
except Exception as e:
    print(f"⚠️ Lookup failed, using empty dict: {e}")
    term_lookup = {}

# ---------------------------------------------------------
# STEP 3: LIST AND DOWNLOAD FILES
# ---------------------------------------------------------
storage_client = storage.Client(project=project_id)
bucket = storage_client.bucket(bucket_name)
blobs = list(bucket.list_blobs(prefix=prefix))
csv_files = [blob.name for blob in blobs if blob.name.endswith('.csv')]

if not csv_files:
    print("❌ No CSV files found in the specified path.")
else:
    for source_file in csv_files:
        print(f"\n🚀 Processing file: {source_file}")
        local_csv = "/tmp/source_tiktok.csv"
        bucket.blob(source_file).download_to_filename(local_csv)

        # TikTok specific: Remove the 'Total' summary row at the bottom
        try:
            df = pd.read_csv(local_csv, engine="python", on_bad_lines='skip')
            df = df.iloc[:-1].copy()
        except Exception as e:
            print(f"❌ Error reading file {source_file}: {e}")
            continue

        # ---------------------------------------------------------
        # STEP 4: ROBUST EXTRACTION & CLEANING
        # ---------------------------------------------------------
        def extract_tiktok_show(row):
            # Tokenize across both Ad and Campaign names using -, _, or space
            search_text = f"{row.get('Ad name', '')} {row.get('Campaign name', '')}"
            parts = re.split(r'[_\-\s]+', str(search_text))

            for part in parts:
                clean_part = part.strip().lower()
                if clean_part in term_lookup:
                    # Priority: Skip generic branding if a specific show term exists
                    if term_lookup[clean_part] == "AJ360 Branding" and len(parts) > 1:
                        continue
                    return term_lookup[clean_part]
            return "AJ360 Branding"

        df['show_title'] = df.apply(extract_tiktok_show, axis=1)

        # Standardize column names (lower_snake_case)
        df.columns = [re.sub(r'[^a-zA-Z0-9_]', '', col.lower().replace(' ', '_').replace('/', '_')) for col in df.columns]

        # Robust string cleaning (remove newlines/quotes) and handle nulls
        df = df.apply(lambda x: x.str.strip().replace({r'\r': '', r'\n': '', '"': ''}, regex=True) if x.dtype == "object" else x)
        df = df.replace(['nan', 'NaN', 'None', '--'], np.nan)

        # ---------------------------------------------------------
        # STEP 5: AUTOMATED SCHEMA EVOLUTION (The Fix)
        # ---------------------------------------------------------
        # Pull current schema as the source of truth
        target_table_obj = bq_client.get_table(table_ref)
        current_schema = list(target_table_obj.schema)
        schema_col_names = [field.name for field in current_schema]

        # Identify columns in CSV not present in the BigQuery table
        new_cols = [col for col in df.columns if col not in schema_col_names]

        if new_cols:
            print(f"✨ Detected {len(new_cols)} new columns. Updating BigQuery schema...")
            for col in new_cols:
                # Add new fields as STRING to the schema list
                current_schema.append(bigquery.SchemaField(col, "STRING"))

            # Apply the updated schema to the table in BigQuery
            target_table_obj.schema = current_schema
            bq_client.update_table(target_table_obj, ["schema"])
            print(f"✅ Table updated with new columns: {new_cols}")

            # Refresh our column name list to include the new additions
            schema_col_names = [field.name for field in current_schema]

        # Fill any missing table columns in the DataFrame with None for alignment
        for col in schema_col_names:
            if col not in df.columns:
                df[col] = None

        # Reorder DataFrame to match the BigQuery schema exactly
        df = df[schema_col_names]

        # ---------------------------------------------------------
        # STEP 6: PRODUCTION LOAD (WRITE_APPEND)
        # ---------------------------------------------------------
        # Uses continuous appending strategy supported by the backup mechanism
        job_config = bigquery.LoadJobConfig(
            schema=current_schema,
            write_disposition="WRITE_APPEND",
            source_format="CSV",
            allow_quoted_newlines=True,
            allow_jagged_rows=True,
            max_bad_records=10
        )

        try:
            job = bq_client.load_table_from_dataframe(df, table_ref, job_config=job_config)
            job.result()
            print(f"✅ Successfully appended {len(df)} rows to {table_id}.")
        except Exception as e:
            print(f"❌ Failed to load {source_file}: {e}")

    # Final Cleanup of local temporary file
    if os.path.exists(local_csv):
        os.remove(local_csv)

print("\n✨ Finished processing all files.")

✅ Monthly backup created/updated: tiktok_ads_backup_2026_03_08
✅ Lookup table loaded: 690 terms.

🚀 Processing file: TIKTOK_ADS/tiktok_ads_country_5-7Mar2026.csv
✅ Successfully appended 136 rows to tiktok_ads.

✨ Finished processing all files.


In [2]:
import pandas as pd
import numpy as np
from google.cloud import storage, bigquery
from google.colab import auth
from datetime import datetime
import re
import os

# 1. AUTHENTICATION & CONFIGURATION
auth.authenticate_user()
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_id = "aj360"
table_id = "tiktok_ads_by_country"
bucket_name = "aj360_data"
# Ensure the prefix targets the specific file path in GCS
prefix = "TIKTOK_ADS/tiktok_ads_country_5-7Mar2026.csv"

bq_client = bigquery.Client(project=project_id)
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# ---------------------------------------------------------
# STEP 1: MONTHLY BACKUP MECHANISM
# ---------------------------------------------------------
# Creates a snapshot of the table before any new data is appended
backup_suffix = datetime.now().strftime('%Y_%m_%d').lower()
backup_table_id = f"{table_id}_backup_{backup_suffix}"
backup_ref = f"{project_id}.{dataset_id}.{backup_table_id}"

try:
    bq_client.get_table(table_ref)
    copy_job = bq_client.copy_table(
        table_ref,
        backup_ref,
        job_config=bigquery.CopyJobConfig(write_disposition="WRITE_TRUNCATE")
    )
    copy_job.result()
    print(f"✅ Monthly backup created/updated: {backup_table_id}")
except Exception as e:
    print(f"⚠️ Backup skipped (Table may not exist yet): {e}")

# ---------------------------------------------------------
# STEP 2: LOAD LOOKUP TABLE
# ---------------------------------------------------------
try:
    lookup_query = f"SELECT term, show_title_en FROM `{project_id}.{dataset_id}.paid_ads_shows_lookup`"
    lookup_df = bq_client.query(lookup_query).to_dataframe()
    # Normalize lookup terms to lowercase for case-insensitive matching
    term_lookup = {str(k).lower(): v for k, v in zip(lookup_df['term'], lookup_df['show_title_en'])}
    print(f"✅ Lookup table loaded: {len(term_lookup)} terms.")
except Exception as e:
    print(f"⚠️ Lookup failed, using empty dict: {e}")
    term_lookup = {}

# ---------------------------------------------------------
# STEP 3: LIST AND DOWNLOAD FILES
# ---------------------------------------------------------
storage_client = storage.Client(project=project_id)
bucket = storage_client.bucket(bucket_name)
blobs = list(bucket.list_blobs(prefix=prefix))
csv_files = [blob.name for blob in blobs if blob.name.endswith('.csv')]

if not csv_files:
    print("❌ No CSV files found in the specified path.")
else:
    for source_file in csv_files:
        print(f"\n🚀 Processing file: {source_file}")
        local_csv = "/tmp/source_tiktok.csv"
        bucket.blob(source_file).download_to_filename(local_csv)

        # TikTok specific: Remove the 'Total' summary row at the bottom
        try:
            df = pd.read_csv(local_csv, engine="python", on_bad_lines='skip')
            df = df.iloc[:-1].copy()
        except Exception as e:
            print(f"❌ Error reading file {source_file}: {e}")
            continue

        # ---------------------------------------------------------
        # STEP 4: ROBUST EXTRACTION & CLEANING
        # ---------------------------------------------------------
        def extract_tiktok_show(row):
            # Tokenize across both Ad and Campaign names using -, _, or space
            search_text = f"{row.get('Ad name', '')} {row.get('Campaign name', '')}"
            parts = re.split(r'[_\-\s]+', str(search_text))

            for part in parts:
                clean_part = part.strip().lower()
                if clean_part in term_lookup:
                    # Priority: Skip generic branding if a specific show term exists
                    if term_lookup[clean_part] == "AJ360 Branding" and len(parts) > 1:
                        continue
                    return term_lookup[clean_part]
            return "AJ360 Branding"

        df['show_title'] = df.apply(extract_tiktok_show, axis=1)

        # Standardize column names (lower_snake_case)
        df.columns = [re.sub(r'[^a-zA-Z0-9_]', '', col.lower().replace(' ', '_').replace('/', '_')) for col in df.columns]

        # Robust string cleaning (remove newlines/quotes) and handle nulls
        df = df.apply(lambda x: x.str.strip().replace({r'\r': '', r'\n': '', '"': ''}, regex=True) if x.dtype == "object" else x)
        df = df.replace(['nan', 'NaN', 'None', '--'], np.nan)

        # ---------------------------------------------------------
        # STEP 5: AUTOMATED SCHEMA EVOLUTION (The Fix)
        # ---------------------------------------------------------
        # Pull current schema as the source of truth
        target_table_obj = bq_client.get_table(table_ref)
        current_schema = list(target_table_obj.schema)
        schema_col_names = [field.name for field in current_schema]

        # Identify columns in CSV not present in the BigQuery table
        new_cols = [col for col in df.columns if col not in schema_col_names]

        if new_cols:
            print(f"✨ Detected {len(new_cols)} new columns. Updating BigQuery schema...")
            for col in new_cols:
                # Add new fields as STRING to the schema list
                current_schema.append(bigquery.SchemaField(col, "STRING"))

            # Apply the updated schema to the table in BigQuery
            target_table_obj.schema = current_schema
            bq_client.update_table(target_table_obj, ["schema"])
            print(f"✅ Table updated with new columns: {new_cols}")

            # Refresh our column name list to include the new additions
            schema_col_names = [field.name for field in current_schema]

        # Fill any missing table columns in the DataFrame with None for alignment
        for col in schema_col_names:
            if col not in df.columns:
                df[col] = None

        # Reorder DataFrame to match the BigQuery schema exactly
        df = df[schema_col_names]

        # ---------------------------------------------------------
        # STEP 6: PRODUCTION LOAD (WRITE_APPEND)
        # ---------------------------------------------------------
        # Uses continuous appending strategy supported by the backup mechanism
        job_config = bigquery.LoadJobConfig(
            schema=current_schema,
            write_disposition="WRITE_APPEND",
            source_format="CSV",
            allow_quoted_newlines=True,
            allow_jagged_rows=True,
            max_bad_records=10
        )

        try:
            job = bq_client.load_table_from_dataframe(df, table_ref, job_config=job_config)
            job.result()
            print(f"✅ Successfully appended {len(df)} rows to {table_id}.")
        except Exception as e:
            print(f"❌ Failed to load {source_file}: {e}")

    # Final Cleanup of local temporary file
    if os.path.exists(local_csv):
        os.remove(local_csv)

print("\n✨ Finished processing all files.")

✅ Monthly backup created/updated: tiktok_ads_by_country_backup_2026_03_08
✅ Lookup table loaded: 690 terms.

🚀 Processing file: TIKTOK_ADS/tiktok_ads_country_5-7Mar2026.csv
✅ Successfully appended 136 rows to tiktok_ads_by_country.

✨ Finished processing all files.
